## Modèle de Machine Learning : 
### 1. Logistic Regression ( C=1 )

In [ ]:
import pandas as pd

from sklearn.pipeline import Pipeline
# Un Pipeline permet de regrouper et organiser plusieurs
# étapes de traitement dans un seul objet et execute dans un ordre
from sklearn.preprocessing import StandardScaler
# stantardiser les variables numeriques pour les mettre sur la meme echelle
from sklearn.linear_model import LogisticRegression

import sys
import os

sys.path.append(os.path.abspath(".."))

from src.evaluation import evaluate_cv


# Charger les données d'entraînement
train_final = pd.read_csv("../data/train_final.csv")

# Séparer les variables explicatives et la cible
X = train_final.drop("Purchased", axis=1)
y = train_final["Purchased"]

# Vérifier les données
print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

# Vérifier les types des colonnes
print(X.dtypes)

Dimensions de X : (7020, 17)
Dimensions de y : (7020,)
AdminPages           int64
AdminTimeSec       float64
InfoPages            int64
InfoTimeSec        float64
CatalogPages         int64
CatalogTimeSec     float64
BounceRate         float64
ExitRate           float64
EngagementValue    float64
PromoProximity     float64
CampaignType           str
DeviceOS             int64
BrowserCode          int64
GeoZone              int64
EntryChannel         int64
CustomerType           str
Weekend              int64
dtype: object


Nous avons 17 variables explicatives :
- 10 variables numériques continues ou quantitatives (standardisation) ---> StandarScaler
- 2 variables catégorielles textuelles (Encodage)
- 4 variables codées numériquement, mais qui représentent des catégories (Encodage) ---> OneHotEncoder
- 1 variable binaire (Weekend) (garde 0/1)

In [ ]:
from sklearn.compose import ColumnTransformer
# un outil qui permet d'appliquer des traitements différents selon les colonnes
from sklearn.preprocessing import StandardScaler, OneHotEncoder
# OneHotEncoder permet de transformer les catégories en colonnes numériques
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# 1. Définir les variables numériques
numeric_features = [
    "AdminPages", "AdminTimeSec",
    "InfoPages", "InfoTimeSec",
    "CatalogPages", "CatalogTimeSec",
    "BounceRate", "ExitRate",
    "EngagementValue", "PromoProximity"
]

# 2. Définir les variables catégorielles
categorical_features = [
    "CampaignType", "CustomerType",
    "DeviceOS", "BrowserCode",
    "GeoZone", "EntryChannel"
]

# 3. Préparer le prétraitement
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ],
    # les autres colonnes non spécifiées dans numeric_features 
    # et categorical_features seront laissées inchangées
    remainder="passthrough"
)

# 4. Construire le pipeline complet en deux etapes :
# 1. le prétraitement des données
# 2. l'entrainement du modele de regression logistique
model = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
    # max_iter=1000 pour effectuer jusqu'à 1000 itérations pour trouver les parametres de modele 
])

In [7]:
# Évaluer le modèle avec 5-fold cross-validation
results = evaluate_cv(model, X, y)

# Afficher les résultats de chaque fold
print(results)

# Calculer la moyenne des métriques
print("\nMoyenne des métriques :")
print(results.drop(columns="Fold").mean())

   Fold  Precision    Recall        F1        AP
0     1   0.593220  0.162791  0.255474  0.385917
1     2   0.594203  0.190698  0.288732  0.391055
2     3   0.603774  0.148837  0.238806  0.413574
3     4   0.620690  0.166667  0.262774  0.421348
4     5   0.548387  0.157407  0.244604  0.412129

Moyenne des métriques :
Precision    0.592055
Recall       0.165280
F1           0.258078
AP           0.404805
dtype: float64


#### Évaluation initiale — Logistic Regression

Une première évaluation du modèle Logistic Regression a été réalisée avec une validation croisée Stratified K-Fold à 5 folds, sur le jeu de données `train_final.csv`.

Les résultats moyens obtenus sont :

- Precision : 0,5921
  * Parmi les visiteurs que le modèle prédit comme acheteurs, environ 59,21 % sont réellement des acheteurs.
- Recall : 0,1653
  * Le modèle détecte seulement 16,53 % des véritables acheteurs.
- F1-score : 0,2581
- Average Precision : 0,4048
  * L'Average Precision évalue la capacité du modèle à classer les visiteurs en plaçant les acheteurs avant les non-acheteurs.

Le modèle présente une précision relativement correcte, mais son Recall reste faible. Il détecte donc une proportion limitée des véritables acheteurs.

Cette difficulté peut notamment être liée au déséquilibre des classes dans le dataset (0 = 85% et 1 = 15%) et aussi Logistic Regression utilise généralement un seuil de décision de 0.5.

### 2. Modèle de Machine Learning : Logistic Regression avec class_weight="balanced"

In [8]:
# Construire le deuxième modèle avec équilibrage des classes
# class_weight="balanced" donne automatiquement un poids plus important a  la classe minoritaire 1
model_balanced = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

In [9]:
# Évaluer le modèle avec 5-fold cross-validation
results_balanced = evaluate_cv(model_balanced, X, y)

# Afficher les résultats de chaque fold
print(results_balanced)

# Calculer les moyennes
print("\nMoyenne des métriques :")
print(results_balanced.drop(columns="Fold").mean())

   Fold  Precision    Recall        F1        AP
0     1   0.292994  0.641860  0.402332  0.392024
1     2   0.324201  0.660465  0.434916  0.392330
2     3   0.334140  0.641860  0.439490  0.423433
3     4   0.308235  0.606481  0.408736  0.421993
4     5   0.310268  0.643519  0.418675  0.420410

Moyenne des métriques :
Precision    0.313968
Recall       0.638837
F1           0.420830
AP           0.410038
dtype: float64


In [10]:
comparison = pd.DataFrame({
    "Logistic Regression": results.drop(columns="Fold").mean(),
    "Logistic Regression Balanced": results_balanced.drop(columns="Fold").mean()
})

print(comparison)

           Logistic Regression  Logistic Regression Balanced
Precision             0.592055                      0.313968
Recall                0.165280                      0.638837
F1                    0.258078                      0.420830
AP                    0.404805                      0.410038


A. Precision : elle diminue
- Modèle classique : 59,21 %
- Modèle Balanced : 31,40 %

Le modèle Balanced fait davantage de prédictions positives, mais une partie de ces prédictions concerne des visiteurs qui n'achètent pas réellement.

B. Recall : il augmente fortement
- Modèle classique : 16,53 %
- Modèle Balanced : 63,88 %

Le modèle Balanced détecte maintenant environ 64 % des véritables acheteurs, contre seulement 16,5 % pour le modèle classique.

C. F1-score : il augmente
- Modèle classique : 25,81 %
- Modèle Balanced : 42,08 %

Le modèle Balanced offre un meilleur équilibre entre Precision et Recall.

D. AP : légère amélioration
- Modèle classique : 0,4048
- Modèle Balanced : 0,4100

### 3. Comparaison entre Logistic Regression et Logistic Regression Balanced

L'ajout de `class_weight="balanced"` a permis d'améliorer considérablement le Recall, qui passe de 0,1653 à 0,6388. Le F1-score augmente également de 0,2581 à 0,4208.

Cependant, cette amélioration s'accompagne d'une diminution de la Precision, qui passe de 0,5921 à 0,3140.

Concernant notre métrique principale, l'Average Precision (AP), le modèle Balanced obtient un résultat légèrement supérieur (0,4100 contre 0,4048).

Ainsi, la prise en compte du déséquilibre des classes améliore la détection des acheteurs, mais son effet sur la qualité globale du classement reste limité.

### 4. La regularisation : tester plusieurs des valeurs de c 

In [11]:
# Les différentes valeurs de C à tester
C_values = [0.01, 0.1, 1, 10, 100]

# Stocker les résultats
results_C = []

# Tester chaque valeur de C
for c in C_values:

    # Créer le modèle Logistic Regression
    model_C = Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", LogisticRegression(
            C=c,
            max_iter=1000,
            class_weight="balanced"
        ))
    ])

    # Évaluer le modèle avec la validation croisée
    scores = evaluate_cv(model_C, X, y)

    # Calculer la moyenne des métriques
    mean_scores = scores.drop(columns="Fold").mean()

    # Ajouter les résultats
    results_C.append({
        "C": c,
        "Precision": mean_scores["Precision"],
        "Recall": mean_scores["Recall"],
        "F1": mean_scores["F1"],
        "AP": mean_scores["AP"]
    })

# Afficher le tableau comparatif
comparison_C = pd.DataFrame(results_C)

print(comparison_C)

        C  Precision    Recall        F1        AP
0    0.01   0.331824  0.617472  0.431413  0.416922
1    0.10   0.320014  0.631404  0.424536  0.413947
2    1.00   0.313968  0.638837  0.420830  0.410038
3   10.00   0.312800  0.636051  0.419226  0.409006
4  100.00   0.312868  0.636981  0.419479  0.408828


Interprétation des résultats

A. Precision

La meilleure Precision est obtenue avec C = 0.01 (33,18 %).

Quand C augmente, la Precision diminue globalement.

Cela signifie que la configuration avec une régularisation plus forte produit ici moins de fausses prédictions positives.

B. Recall

Le meilleur Recall est obtenu avec C = 1 (63,88 %).

Les cinq configurations ont des Recall assez proches : entre 61,75 % et 63,88 %.

Cela signifie que le changement de C influence relativement peu la détection des acheteurs.

C. F1-score

Le meilleur F1-score est obtenu avec C = 0.01 (43,14 %).

Cette configuration offre le meilleur équilibre entre Precision et Recall parmi les valeurs testées.

D. Average Precision (AP)

Meilleur résultat
La meilleure AP est obtenue avec C = 0.01 :
- C = 0.01 → AP = 0.4169
- C = 0.1 → AP = 0.4139
- C = 1 → AP = 0.4100
- C = 10 → AP = 0.4090
- C = 100 → AP = 0.4088
On constate que l'AP diminue progressivement lorsque C augmente.

Cela indique que, parmi les configurations testées, la régularisation plus forte avec C = 0.01 donne le meilleur classement des acheteurs.

### conclustion 

Afin d'étudier l'influence de la régularisation, plusieurs valeurs du paramètre C ont été testées sur le modèle Logistic Regression avec class_weight="balanced" : 0.01, 0.1, 1, 10 et 100.

Les résultats montrent que la configuration avec C = 0.01 obtient les meilleures performances selon l'Average Precision (AP), avec une valeur de 0.4169, ainsi que le meilleur F1-score (0.4314).

Lorsque C augmente, l'AP diminue progressivement, passant de 0.4169 à 0.4088.

### 5. Comparaison globale des configurations
* Logistic Regression classique (c =1 )
* Logistic Regression Balanced (c = 1)
* Logistic Regression Balanced (c = 0.01)

In [15]:
# Tableau de comparaison des configurations
comparison_final = pd.DataFrame({
    "LR classique": results.drop(columns="Fold").mean(),
    "LR balanced (C=1)": results_balanced.drop(columns="Fold").mean(),
    "LR balanced (C=0.01)": (
        # selectionne la valeur de la ligne a une valeur c = 0.01
        # retourne true / false 
        comparison_C[comparison_C["C"] == 0.01]
        # selectionner les colonnes 
        [["Precision", "Recall", "F1", "AP"]]
        # recuperer la ligne sous forme SeriePandas
        .iloc[0]
    )
})

print(comparison_final)

comparison_final.loc["AP"].sort_values(ascending=False)

           LR classique  LR balanced (C=1)  LR balanced (C=0.01)
Precision      0.592055           0.313968              0.331824
Recall         0.165280           0.638837              0.617472
F1             0.258078           0.420830              0.431413
AP             0.404805           0.410038              0.416922


LR balanced (C=0.01)    0.416922
LR balanced (C=1)       0.410038
LR classique            0.404805
Name: AP, dtype: float64

Interprétation de chaque métrique

A. Precision

La meilleure Precision est obtenue par Logistic Regression classique : 59,21 %.

Cela signifie que parmi les visiteurs prédits comme acheteurs, environ 59 % sont réellement des acheteurs.

Le modèle Balanced obtient une Precision plus faible, car il prédit davantage de visiteurs comme acheteurs.

B. Recall

Le meilleur Recall est obtenu par Logistic Regression Balanced avec C=1 : 63,88 %.

Cela signifie qu'il détecte environ 64 % des véritables acheteurs.

En comparaison, le modèle classique n'en détecte que 16,53 %.

C. F1-score

Le meilleur F1-score est obtenu avec Balanced et C=0.01 : 43,14 %.

Cette configuration offre le meilleur équilibre entre Precision et Recall parmi les trois configurations.

D. Average Precision (AP) ( metrique principale ) : 

- Balanced (C=0.01) : 0.4169.
- Balanced (C=1) : 0.4100.
- Classique : 0.4048.

La configuration Balanced avec C=0.01 obtient donc le meilleur résultat selon notre métrique principale.

#### Comparaison des configurations de Logistic Regression

Trois configurations de Logistic Regression ont été comparées à l'aide d'une validation croisée Stratified K-Fold à 5 folds.

Les résultats montrent que le modèle classique possède la meilleure Precision (0,5921), mais présente un Recall faible (0,1653).

L'utilisation de `class_weight="balanced"` améliore considérablement le Recall. Ensuite, le réglage du paramètre C permet d'obtenir de meilleures performances.

La configuration Logistic Regression avec `class_weight="balanced"` et C = 0,01 obtient le meilleur Average Precision (0,4169) et le meilleur F1-score (0,4314).

Ainsi, parmi les configurations testées, cette dernière est retenue comme meilleure configuration linéaire selon notre métrique principale, l'Average Precision.

### 6. Évaluation du modèle SVM linéaire (LinearSVC)

In [17]:
from sklearn.svm import LinearSVC

In [18]:
model_svm = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LinearSVC(
        C=1,
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    ))
])

In [19]:
results_svm = evaluate_cv(model_svm, X, y)

print(results_svm)

print("\nMoyenne des métriques :")
print(results_svm.drop(columns="Fold").mean())

   Fold  Precision    Recall        F1        AP
0     1   0.292473  0.632558  0.400000  0.387961
1     2   0.326389  0.655814  0.435858  0.389383
2     3   0.334146  0.637209  0.438400  0.418806
3     4   0.302108  0.597222  0.401244  0.420656
4     5   0.306306  0.629630  0.412121  0.413454

Moyenne des métriques :
Precision    0.312284
Recall       0.630487
F1           0.417525
AP           0.406052
dtype: float64


Interprétation :

- Precision = 31,23 % : 

parmi les clients prédits comme acheteurs, environ 31 % sont réellement acheteurs.
- Recall = 63,05 % : 

le modèle détecte environ 63 % des vrais acheteurs.
- F1-score = 41,75 % : 

équilibre entre Precision et Recall.
- AP = 40,61 % : 

capacité du modèle à classer les acheteurs avant les non-acheteurs.

## 7. Comparaison des modèles linéaires

In [21]:
comparison_models = pd.DataFrame({
    "LR classique": results.drop(columns="Fold").mean(),
    "LR balanced (C=1)": results_balanced.drop(columns="Fold").mean(),
    "LR balanced (C=0.01)": (
        comparison_C[comparison_C["C"] == 0.01]
        [["Precision", "Recall", "F1", "AP"]]
        .iloc[0] ) ,
    "LinearSVC (C=1)": results_svm.drop(columns="Fold").mean()
})

print(comparison_models)

comparison_models.loc["AP"].sort_values(ascending=False)

           LR classique  LR balanced (C=1)  LR balanced (C=0.01)  \
Precision      0.592055           0.313968              0.331824   
Recall         0.165280           0.638837              0.617472   
F1             0.258078           0.420830              0.431413   
AP             0.404805           0.410038              0.416922   

           LinearSVC (C=1)  
Precision         0.312284  
Recall            0.630487  
F1                0.417525  
AP                0.406052  


LR balanced (C=0.01)    0.416922
LR balanced (C=1)       0.410038
LinearSVC (C=1)         0.406052
LR classique            0.404805
Name: AP, dtype: float64

#### Interprétation des résultats

La comparaison montre que la régression logistique équilibrée avec C=0.01
obtient le meilleur résultat selon la métrique AP (0.4169), ainsi que le
meilleur F1-score (0.4314).

La régression logistique classique possède une meilleure Precision,
mais son Recall est faible.

Le modèle LinearSVC obtient un AP de 0.4061, inférieur à celui des deux
configurations de régression logistique équilibrée.

Ainsi, Logistic Regression balanced avec C=0.01 représente actuellement
notre meilleure configuration parmi les modèles linéaires testés.

## 8. Optimisation des paramètres de LinearSVC

In [ ]:
C_values_svm = [0.01, 0.1, 1, 10, 100]

results_svm_C = []

for c in C_values_svm:

    model_svm_C = Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", LinearSVC(
            C=c,
            class_weight="balanced",
            max_iter=5000,
            random_state=42
        ))
    ])

    scores = evaluate_cv(model_svm_C, X, y)

    mean_scores = scores.drop(columns="Fold").mean()

    results_svm_C.append({
        "C": c,
        "Precision": mean_scores["Precision"],
        "Recall": mean_scores["Recall"],
        "F1": mean_scores["F1"],
        "AP": mean_scores["AP"]
    })

comparison_svm_C = pd.DataFrame(results_svm_C)

print(comparison_svm_C)

comparison_svm_C.set_index("C")["AP"].sort_values(ascending=False)
# set_index("c") utilise les valeurs de c comme index 

        C  Precision    Recall        F1        AP
0    0.01   0.320239  0.620263  0.422232  0.410816
1    0.10   0.313248  0.629556  0.418199  0.407231
2    1.00   0.312284  0.630487  0.417525  0.406052
3   10.00   0.312462  0.631417  0.417877  0.405852
4  100.00   0.312462  0.631417  0.417877  0.405848


C
0.01      0.410816
0.10      0.407231
1.00      0.406052
10.00     0.405852
100.00    0.405848
Name: AP, dtype: float64

Interprétation :
- C=0.01 donne le meilleur AP : 0.4108.
- C=0.01 donne également le meilleur F1 et Precision.
- C=10 et C=100 donnent le meilleur Recall, mais leur AP est plus faible.
- Lorsque C augmente, l'AP est légèrement inférieur .

#### Meilleur configuration de Modele SVM: 
LinearSVC(C=0.01, class_weight="balanced")

## 9. Comparaison finale des meilleurs modèles linéaires

In [ ]:
best_linear_models = pd.DataFrame({
    "LR balanced (C=0.01)": (
        comparison_C[comparison_C["C"] == 0.01] # true/false
        [["Precision", "Recall", "F1", "AP"]]
        .iloc[0]
    ),

    "LinearSVC balanced (C=0.01)": (
        comparison_svm_C[comparison_svm_C["C"] == 0.01]
        [["Precision", "Recall", "F1", "AP"]]
        .iloc[0]
    )
})

print(best_linear_models)

best_linear_models.loc["AP"].sort_values(ascending=False)

           LR balanced (C=0.01)  LinearSVC balanced (C=0.01)
Precision              0.331824                     0.320239
Recall                 0.617472                     0.620263
F1                     0.431413                     0.422232
AP                     0.416922                     0.410816


LR balanced (C=0.01)           0.416922
LinearSVC balanced (C=0.01)    0.410816
Name: AP, dtype: float64

### Conclusion de la comparaison des modèles linéaires

La comparaison des deux meilleures configurations montre que
Logistic Regression avec C=0.01 et class_weight="balanced"
obtient le meilleur AP (0.4169) et le meilleur F1-score (0.4314).

LinearSVC avec C=0.01 obtient un Recall légèrement supérieur
(0.6203 contre 0.6175).

* Logistic Regression comme meilleur modèle linéaire
parmi les configurations testées.

## 10. Évaluation finale sur le jeu de validation

In [ ]:
validation = pd.read_csv("../data/validation.csv")

X_val = validation.drop("Purchased", axis=1)
y_val = validation["Purchased"]

print("Dimensions de X_val :", X_val.shape)
print("Dimensions de y_val :", y_val.shape)

Dimensions de X_val : (2340, 17)
Dimensions de y_val : (2340,)


In [34]:
# Entraîner le meilleur modèle Logistic Regression sur les donnees 
# de train_final et évaluer sur les données de validation
best_lr = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(
        C=0.01,
        max_iter=1000,
        class_weight="balanced"
    ))
])

best_lr.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different tr

In [35]:
# Faire les prédictions sur validation
# y_val_pred contient les classes prédites (0 ou 1)
# y_val_scores contient les probabilités estimées d'achat
y_val_pred = best_lr.predict(X_val)

y_val_scores = best_lr.predict_proba(X_val)[:, 1]

In [36]:
# Calculer les métriques sur le jeu de validation
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score
)

validation_results = {
    "Precision": precision_score(y_val, y_val_pred, zero_division=0),
    "Recall": recall_score(y_val, y_val_pred, zero_division=0),
    "F1": f1_score(y_val, y_val_pred, zero_division=0),
    "AP": average_precision_score(y_val, y_val_scores)
}

pd.DataFrame([validation_results])

,Precision,Recall,F1,AP
0,0.330289,0.604457,0.427165,0.390105


1. Interprétation des résultats

Meilleur modèle : Logistic Regression (C=0.01, class_weight="balanced")

* Precision	: 33,03 % - Environ 33 % des clients prédits acheteurs sont réellement acheteurs.
* Recall : 60,45 % - Le modèle détecte environ 60 % des vrais acheteurs.
* F1-score : 42,72 % - Équilibre entre Precision et Recall.
* AP : 39,01 % - Qualité du classement des acheteurs avant les non-acheteurs.

### Comparaison avec la validation croisée

In [ ]:
comparison_cv_validation = pd.DataFrame({
    "Cross-Validation": [0.331824, 0.617472, 0.431413, 0.416922],
    "Validation": [0.330289, 0.604457, 0.427165, 0.390105]
}, index=["Precision", "Recall", "F1", "AP"])

comparison_cv_validation
# afficher les résultats en pourcentage avec deux chiffres après la virgule
(comparison_cv_validation * 100).round(2)


,Cross-Validation,Validation
Precision,33.18,33.03
Recall,61.75,60.45
F1,43.14,42.72
AP,41.69,39.01


Interprétation :

Les résultats sont assez proches entre la validation croisée et le jeu de validation.

L'AP diminue de 41,69 % à 39,01 %, mais cette différence reste modérée. Cela indique que les performances du modèle sont relativement cohérentes sur ces deux évaluations.

### Interprétation des résultats sur le jeu de validation

Le modèle Logistic Regression avec C=0.01 et
class_weight="balanced" obtient un AP de 0.3901 sur
le jeu de validation.

Le Recall est de 0.6045, ce qui signifie que le modèle
détecte environ 60 % des acheteurs réels.

Les résultats sont proches de ceux obtenus avec la
validation croisée, avec une légère diminution de l'AP.

Le modèle présente donc des performances relativement
cohérentes sur les données de validation.

## 11. Bilan des expérimentations des modèles linéaires

In [ ]:
bilan_modeles = pd.DataFrame({
    "Modèle": [
        "Logistic Regression classique",
        "Logistic Regression balanced (C=1)",
        "Logistic Regression balanced (C=0.01)",
        "LinearSVC balanced (C=1)",
        "LinearSVC balanced (C=0.01)"
    ],
    "AP (Cross-Validation)": [
        0.404805,
        0.410038,
        0.416922,
        0.406052,
        0.410816
    ]
})

# tri des modèles par AP décroissante
bilan_modeles = bilan_modeles.sort_values(
    by="AP (Cross-Validation)",
    ascending=False
).reset_index(drop=True)

bilan_modeles

,Modèle,AP (Cross-Validation)
0,Logistic Regression balanced (C=0.01),0.416922
1,LinearSVC balanced (C=0.01),0.410816
2,Logistic Regression balanced (C=1),0.410038
3,LinearSVC balanced (C=1),0.406052
4,Logistic Regression classique,0.404805


#### Interprétation

Ce tableau permet de voir rapidement quelle configuration a obtenu le meilleur AP pendant la validation croisée.

Notre meilleur modèle linéaire reste :

Logistic Regression (C=0.01, class_weight="balanced")

avec un AP de 0.416922.